In [ ]:
import numpy as np
import faiss
import pandas as pd
import os
import sys
sys.path.append('/Users/mac/Documents/Chat-Bot-Telegram/SQL')
import sql_query

In [2]:
#run venv and run it again

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-29 23:42:56.718888: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:

chunk_size = 10000
checkchunk = 0
numberchunk = 0
chunktotal = 0
if os.path.exists('/content/drive/MyDrive/chunks/vectors.npy'):
  chunklast = np.load('/content/drive/MyDrive/chunks/vectors.npy')
  checkchunk = len(chunklast)
  numberchunk = checkchunk//10000
else:
  np.save('/content/drive/MyDrive/chunks/vectors.npy', np.empty((0,384)))

for chunk in pd.read_csv('/content/drive/MyDrive/chunks/all_information_only.csv', chunksize=chunk_size):
    chunktotal += len(chunk)
    if chunktotal <= checkchunk :
        continue
    else:
      vectors = model.encode(chunk['all_information'].reset_index(drop=True), batch_size=32, show_progress_bar=True)
      last_vector = np.load('/content/drive/MyDrive/chunks/vectors.npy')
      print(len(last_vector))
      all_vec = np.vstack([last_vector, vectors])
      np.save('/content/drive/MyDrive/chunks/vectors.npy', all_vec)
      numberchunk += 1
      print(numberchunk)



In [ ]:

chunk_size = 10000
total_chunk = 0
total_file_chunk = 0
vector_file = '/content/drive/MyDrive/chunks/vectors.npy'
main_file = '/content/drive/MyDrive/vector'


existing_files = [f for f in os.listdir(main_file) if f.startswith("vectors_chunk_") and f.endswith(".npy")]
total_file_chunk = len(existing_files) * chunk_size


for  i ,chunk in enumerate(pd.read_csv('/content/drive/MyDrive/chunks/all_information_only.csv', chunksize=chunk_size)):
    total_chunk += len(chunk)
    if total_chunk <= total_file_chunk:
      continue
    else:
      chunk.dropna(inplace=True)
      vectors = model.encode(chunk['all_information'].reset_index(drop=True), batch_size=32, show_progress_bar=True)
      if vectors.ndim != 2 or vectors.shape[1] != 384:
        print(f'file is not sync with 384 vectors{i+1}')
        continue
      output_path = os.path.join(main_file, f'vectors_chunk_{i+1}.npy')
      print(f"✅ Saved chunk {i+1} at {output_path}")
      np.save(output_path,vectors)

In [ ]:

vectors  = np.load('/Users/mac/Documents/Chat-Bot-Telegram/embedding.npy')

for rowtext, vector in zip(chunk['all_information'], vectors):
     sql_query.create('user', rowtext, vector)

In [ ]:
prompt = input("لطفاً سوال خود را درباره محصول وارد کنید: ")

vector = model.encode(prompt)
